# GA4 E-Commerce Funnel Analysis
**Dataset:** `bigquery-public-data.ga4_obfuscated_sample_ecommerce`  
**Stack:** BigQuery (SQL) → Pandas → Plotly  
**Date range:** Nov 2020 – Jan 2021 (full public dataset)

## What this notebook covers

| Section | Analysis |
|---|---|
| 1 | Setup & config |
| 2 | Schema exploration — event inventory, date range |
| 3 | **Session-scoped funnel** — conversion within a single visit |
| 4 | Revenue layer — $ impact at each funnel step |
| 5 | Device category breakdown |
| 6 | Time-to-convert — median hours between steps |
| 7 | Alternative entry: shoppers funnel starting at `view_item` |
| 8 | Weekly cohort retention |
| 9 | Summary & hypotheses |
| 10 | **AI-powered insight generation** — LLM product brief |

> **Session vs user grain:** sections 3–7 use session-scoped funnels (each row = one visit).
> This matches how the GA4 UI reports conversion and gives sharper, more actionable numbers
> than user-scoped funnels where a past purchaser "carries over" into every future session.

## 1. Setup

In [1]:
# ── Section 1: Setup ──────────────────────────────────────────────────────────
# Suppress non-critical library warnings so notebook output stays readable.
import warnings
warnings.filterwarnings('ignore')

# BigQuery: run SQL against GA4 export tables in the cloud.
from google.cloud import bigquery
# Pandas: reshape query results into DataFrames for metrics and charts.
import pandas as pd
# Plotly: interactive funnel, bar, and heatmap visualizations.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# GCP project that owns the BigQuery job (billing + credentials).
PROJECT_ID = "project-92e3923f-0e27-4c05-b11"
# Fully qualified BigQuery dataset: GA4 obfuscated e-commerce public sample.
DATASET    = "bigquery-public-data.ga4_obfuscated_sample_ecommerce"

# Analysis window — matches the full public sample (Nov 2020 – Jan 2021).
# Widen carefully: each extra month adds ~$0.005 per query at this dataset size.
DATE_START = "20201101"
DATE_END   = "20210131"

# Client uses Application Default Credentials (gcloud auth / service account).
client = bigquery.Client(project=PROJECT_ID)
print(f"BigQuery client ready  |  project: {PROJECT_ID}")
print(f"Date window: {DATE_START} → {DATE_END}")


BigQuery client ready  |  project: project-92e3923f-0e27-4c05-b11
Date window: 20201101 → 20210131


## 2. Schema exploration
Confirm date range and event inventory before writing analysis queries.

In [2]:
# ── Section 2a: Dataset date range ───────────────────────────────────────────
# GA4 exports one table per day: events_YYYYMMDD. _TABLE_SUFFIX is the date suffix.
# Scan all daily tables to confirm earliest/latest dates and table count.
date_range_df = client.query(f"""
SELECT
  MIN(_TABLE_SUFFIX) AS earliest_date,
  MAX(_TABLE_SUFFIX) AS latest_date,
  COUNT(DISTINCT _TABLE_SUFFIX) AS num_daily_tables
FROM `{DATASET}.events_*`
""").to_dataframe()

print("Full dataset date range:")
display(date_range_df)


Full dataset date range:


,earliest_date,latest_date,num_daily_tables
0,20201101,20210131,92


In [3]:
# ── Section 2b: Event inventory ────────────────────────────────────────────
# List every event_name in the analysis window with volume and reach.
# Use this to validate funnel step events exist before building session logic.
events_df = client.query(f"""
SELECT
  event_name,
  COUNT(*)                       AS event_count,
  COUNT(DISTINCT user_pseudo_id) AS unique_users
FROM `{DATASET}.events_*`
WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
GROUP BY event_name
ORDER BY event_count DESC
""").to_dataframe()

print(f"Events in window {DATE_START}–{DATE_END}:")
display(events_df)


Events in window 20201101–20210131:


,event_name,event_count,unique_users
0,page_view,1350428,269792
1,user_engagement,1058721,213004
2,scroll,493072,138098
3,view_item,386068,61252
4,session_start,354970,267116
5,first_visit,257462,257314
6,view_promotion,190104,102443
7,add_to_cart,58543,12545
8,begin_checkout,38757,9715
9,select_item,31007,13180


## 3. Session-scoped funnel

**Grain:** one row = one session (`user_pseudo_id` + `ga_session_id`).
A session converts at step N only if it reached step N−1 first *within the same session*.

**Why session-scoped?**
- Matches GA4 UI funnel reports
- Avoids inflating conversion by counting past purchasers who return for unrelated visits
- Step-to-step rates reflect a single coherent user intent

**Funnel:** `session_start` → `view_item` → `add_to_cart` → `begin_checkout` → `purchase`

In [5]:
# ── Section 3: Session-scoped funnel (BigQuery) ─────────────────────────────
# Grain: one row per (user_pseudo_id, ga_session_id). Steps must occur in order
# within the same visit — mirrors GA4 UI funnel reporting.
session_funnel_sql = f"""
-- base: pull funnel events and extract ga_session_id from the event_params array.
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value
     FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id'
     LIMIT 1)           AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),

-- session_events: first occurrence of each event per session (dedupe repeats).
session_events AS (
  SELECT
    user_pseudo_id,
    session_id,
    event_name,
    MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),

-- pivoted: one row per session; columns hold earliest timestamp per step.
pivoted AS (
  SELECT
    user_pseudo_id,
    session_id,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END) AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END) AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END) AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END) AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END) AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
),

-- sequential: count sessions reaching each step only if prior steps happened first.
sequential AS (
  SELECT
    COUNT(*)                                                                         AS step1_sessions,
    COUNTIF(ts_view     IS NOT NULL AND ts_view     >= ts_session)                  AS step2_view_item,
    COUNTIF(ts_cart     IS NOT NULL AND ts_cart     >= ts_view
                                   AND ts_view      >= ts_session)                  AS step3_add_to_cart,
    COUNTIF(ts_checkout IS NOT NULL AND ts_checkout >= ts_cart
                                   AND ts_cart      >= ts_view
                                   AND ts_view      >= ts_session)                  AS step4_begin_checkout,
    COUNTIF(ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout
                                   AND ts_checkout  >= ts_cart
                                   AND ts_cart      >= ts_view
                                   AND ts_view      >= ts_session)                  AS step5_purchase
  FROM pivoted
  WHERE ts_session IS NOT NULL
)

SELECT * FROM sequential
"""

funnel_raw = client.query(session_funnel_sql).to_dataframe()
print("Session-scoped funnel counts:")
display(funnel_raw)


Session-scoped funnel counts:


,step1_sessions,step2_view_item,step3_add_to_cart,step4_begin_checkout,step5_purchase
0,354857,75261,14870,5205,2721


In [8]:
# ── Section 3b: Funnel metrics + Plotly visualization ─────────────────────
# Human-readable step labels aligned with GA4 funnel stage names.
step_labels = [
    "1. Session Start",
    "2. View Item",
    "3. Add to Cart",
    "4. Begin Checkout",
    "5. Purchase",
]
# Column names returned by the sequential CTE in session_funnel_sql.
step_cols = [
    "step1_sessions", "step2_view_item", "step3_add_to_cart",
    "step4_begin_checkout", "step5_purchase",
]

# Build a tidy funnel table for display and downstream LLM context (Section 10).
funnel_df = pd.DataFrame({
    "step":  step_labels,
    "sessions": [int(funnel_raw[c].iloc[0]) for c in step_cols],
})
# step_cvr: conversion from the previous step (% who advanced one stage).
funnel_df["step_cvr"]    = (funnel_df["sessions"] / funnel_df["sessions"].shift(1) * 100).round(1)
# overall_cvr: conversion from session start (% still in funnel at this step).
funnel_df["overall_cvr"] = (funnel_df["sessions"] / funnel_df["sessions"].iloc[0] * 100).round(1)
# drop_off: 100 - step_cvr — share lost between adjacent steps.
funnel_df["drop_off"]    = (100 - funnel_df["step_cvr"]).round(1)

print("Session funnel with conversion rates:")
display(funnel_df)

# Funnel chart: width of each stage = remaining sessions.
fig = px.funnel(
    funnel_df, x="sessions", y="step",
    title=f"GA4 Session-Scoped Funnel  ({DATE_START[:6]} – {DATE_END[:6]})",
    color_discrete_sequence=["#4C78A8"],
)
# Annotate each transition with drop-off % and step conversion (skip top of funnel).
for i, row in funnel_df.iterrows():
    if i == 0:
        continue
    fig.add_annotation(
        x=row["sessions"], y=row["step"],
        text=f"  ↓ {row['drop_off']}% drop-off  ({row['step_cvr']}% pass)",
        showarrow=False, xanchor="left",
        font=dict(size=11, color="#d62728"),
    )
fig.update_layout(height=450, margin=dict(l=160, r=260, t=60, b=40))
fig.show()


Session funnel with conversion rates:


,step,sessions,step_cvr,overall_cvr,drop_off
0,1. Session Start,354857,NaN,100.0,NaN
1,2. View Item,75261,21.2,21.2,78.8
2,3. Add to Cart,14870,19.8,4.2,80.2
3,4. Begin Checkout,5205,35.0,1.5,65.0
4,5. Purchase,2721,52.3,0.8,47.7


## 4. Revenue layer
Add `ecommerce.purchase_revenue` to see the **dollar impact** at the purchase step.
Also computes revenue per session entering the funnel and revenue per converting session.

In [9]:
# ── Section 4: Revenue layer ────────────────────────────────────────────────
# Restrict to sessions that completed the full sequential funnel, then sum
# ecommerce.purchase_revenue (only populated on purchase events).
revenue_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp,
    -- Revenue only present on purchase events
    ecommerce.purchase_revenue AS purchase_revenue
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT
    user_pseudo_id, session_id, event_name,
    MIN(event_timestamp)   AS first_ts,
    SUM(purchase_revenue)  AS session_revenue
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END)      AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END)      AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END)      AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END)      AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END)      AS ts_purchase,
    MAX(CASE WHEN event_name = 'purchase'       THEN session_revenue END) AS revenue
  FROM session_events
  GROUP BY user_pseudo_id, session_id
),
-- converting_sessions: same sequential rules as Section 3 funnel.
converting_sessions AS (
  SELECT *
  FROM pivoted
  WHERE
    ts_session  IS NOT NULL
    AND ts_view     >= ts_session
    AND ts_cart     >= ts_view
    AND ts_checkout >= ts_cart
    AND ts_purchase >= ts_checkout
)
SELECT
  COUNT(*)                                AS purchasing_sessions,
  ROUND(SUM(revenue), 2)                  AS total_revenue,
  ROUND(AVG(revenue), 2)                  AS avg_order_value,
  ROUND(APPROX_QUANTILES(revenue, 100)[OFFSET(50)], 2) AS median_order_value,
  ROUND(MIN(revenue), 2)                  AS min_order_value,
  ROUND(MAX(revenue), 2)                  AS max_order_value
FROM converting_sessions
WHERE revenue IS NOT NULL AND revenue > 0
"""

revenue_df = client.query(revenue_sql).to_dataframe()
print("Revenue summary for purchasing sessions:")
display(revenue_df)

# Normalize revenue against all sessions that entered the funnel (step 1).
total_sessions = int(funnel_raw["step1_sessions"].iloc[0])
total_revenue  = float(revenue_df["total_revenue"].iloc[0])
print(f"
Revenue per session entering funnel: ${total_revenue / total_sessions:.4f}")
print(f"Overall session-to-purchase rate:    {int(revenue_df['purchasing_sessions'].iloc[0]) / total_sessions * 100:.2f}%")


Revenue summary for purchasing sessions:


,purchasing_sessions,total_revenue,avg_order_value,median_order_value,min_order_value,max_order_value
0,2461,193678.0,78.7,55.0,3.0,1530.0



Revenue per session entering funnel: $0.5458
Overall session-to-purchase rate:    0.69%


## 5. Device category breakdown
Same session-scoped funnel, split by `device.category`.
Highlights whether mobile vs desktop differences are meaningful or within noise.

In [10]:
# ── Section 5: Device category breakdown (BigQuery) ─────────────────────────
# Same session-scoped sequential funnel as Section 3, grouped by device.category.
device_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    device.category AS device_category,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT
    user_pseudo_id, session_id,
    ANY_VALUE(device_category) AS device_category,
    event_name,
    MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    ANY_VALUE(device_category)                                                   AS device_category,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END)               AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END)               AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END)               AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END)               AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END)               AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
)
SELECT
  device_category,
  COUNT(*)                                                                        AS step1_sessions,
  COUNTIF(ts_view     IS NOT NULL AND ts_view     >= ts_session)                 AS step2_view_item,
  COUNTIF(ts_cart     IS NOT NULL AND ts_cart     >= ts_view
                                 AND ts_view      >= ts_session)                 AS step3_add_to_cart,
  COUNTIF(ts_checkout IS NOT NULL AND ts_checkout >= ts_cart
                                 AND ts_cart      >= ts_view
                                 AND ts_view      >= ts_session)                 AS step4_begin_checkout,
  COUNTIF(ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout
                                 AND ts_checkout  >= ts_cart
                                 AND ts_cart      >= ts_view
                                 AND ts_view      >= ts_session)                 AS step5_purchase
FROM pivoted
WHERE ts_session IS NOT NULL
GROUP BY device_category
ORDER BY step1_sessions DESC
"""

device_raw = client.query(device_sql).to_dataframe()
display(device_raw)


,device_category,step1_sessions,step2_view_item,step3_add_to_cart,step4_begin_checkout,step5_purchase
0,desktop,205899,43810,8606,2985,1537
1,mobile,141079,29793,5948,2108,1123
2,tablet,7879,1658,316,112,61


In [11]:
# ── Section 5b: Device funnel chart ───────────────────────────────────────
# Map wide SQL column names to display labels for melt/pivot operations.
step_map = {
    "step1_sessions":       "1. Session Start",
    "step2_view_item":      "2. View Item",
    "step3_add_to_cart":    "3. Add to Cart",
    "step4_begin_checkout": "4. Begin Checkout",
    "step5_purchase":       "5. Purchase",
}
# Long format: one row per (device, funnel step) for grouped bar chart.
device_long = device_raw.melt(
    id_vars="device_category",
    value_vars=list(step_map.keys()),
    var_name="step_col", value_name="sessions",
)
device_long["step"] = device_long["step_col"].map(step_map)
# Denominator for % conversion: sessions at step 1 per device.
base = (
    device_long[device_long["step_col"] == "step1_sessions"]
    [["device_category","sessions"]].rename(columns={"sessions":"base"})
)
device_long = device_long.merge(base, on="device_category")
device_long["pct"] = (device_long["sessions"] / device_long["base"] * 100).round(2)

# Grouped bar: % of each device's sessions still in funnel at each step.
fig_dev = px.bar(
    device_long, x="step", y="pct", color="device_category",
    barmode="group",
    title="Session Funnel Conversion Rate by Device",
    labels={"pct": "% of Sessions", "step": "", "device_category": "Device"},
    color_discrete_map={"desktop": "#4C78A8", "mobile": "#F58518", "tablet": "#54A24B"},
    text="pct",
)
fig_dev.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig_dev.update_layout(height=450, xaxis_tickangle=-20, uniformtext_minsize=8)
fig_dev.show()

# Absolute purchase counts and session-to-purchase % by device (LLM context uses this).
purchase_rates = device_long[device_long["step_col"] == "step5_purchase"][["device_category","sessions","base","pct"]]
print("
Session-to-purchase rate by device:")
display(purchase_rates.reset_index(drop=True))



Session-to-purchase rate by device:


,device_category,sessions,base,pct
0,desktop,1537,205899,0.75
1,mobile,1123,141079,0.8
2,tablet,61,7879,0.77


## 6. Time-to-convert
For sessions that **complete the full funnel**, compute the median time (in hours)
between each consecutive step. This tells you how long the purchase consideration cycle is
and where the longest pauses happen.

In [12]:
# ── Section 6: Time-to-convert (BigQuery) ─────────────────────────────────
# For sessions that completed the full funnel, compute median hours between steps.
# GA4 event_timestamp is microseconds since epoch; divide by 3.6e12 for hours.
ttc_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('session_start','view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT user_pseudo_id, session_id, event_name, MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    MAX(CASE WHEN event_name = 'session_start'  THEN first_ts END) AS ts_session,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END) AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END) AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END) AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END) AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
),
-- full_funnel: only sessions with valid sequential timestamps through purchase.
full_funnel AS (
  SELECT *
  FROM pivoted
  WHERE
    ts_session  IS NOT NULL
    AND ts_view     >= ts_session
    AND ts_cart     >= ts_view
    AND ts_checkout >= ts_cart
    AND ts_purchase >= ts_checkout
)
SELECT
  -- Gaps in microseconds → convert to hours (p50 = median via APPROX_QUANTILES).
  ROUND(APPROX_QUANTILES((ts_view     - ts_session)  / 3600000000.0, 100)[OFFSET(50)], 2) AS p50_session_to_view_hrs,
  ROUND(APPROX_QUANTILES((ts_cart     - ts_view)     / 3600000000.0, 100)[OFFSET(50)], 2) AS p50_view_to_cart_hrs,
  ROUND(APPROX_QUANTILES((ts_checkout - ts_cart)     / 3600000000.0, 100)[OFFSET(50)], 2) AS p50_cart_to_checkout_hrs,
  ROUND(APPROX_QUANTILES((ts_purchase - ts_checkout) / 3600000000.0, 100)[OFFSET(50)], 2) AS p50_checkout_to_purchase_hrs,
  ROUND(APPROX_QUANTILES((ts_purchase - ts_session)  / 3600000000.0, 100)[OFFSET(50)], 2) AS p50_total_hrs,
  COUNT(*) AS converting_sessions
FROM full_funnel
"""

ttc_df = client.query(ttc_sql).to_dataframe()
print("Median time between funnel steps (hours) — converting sessions only:")
display(ttc_df.T.rename(columns={0: "median_hours"}))


Median time between funnel steps (hours) — converting sessions only:


,median_hours
p50_session_to_view_hrs,0.02
p50_view_to_cart_hrs,0.0
p50_cart_to_checkout_hrs,0.11
p50_checkout_to_purchase_hrs,0.06
p50_total_hrs,0.26
converting_sessions,2721


In [13]:
# ── Section 6b: Time-to-convert bar chart ─────────────────────────────────
# Reshape median hour columns into a simple table for Plotly.
ttc_steps = pd.DataFrame({
    "transition": [
        "Session → View Item",
        "View Item → Add to Cart",
        "Add to Cart → Checkout",
        "Checkout → Purchase",
    ],
    "median_hours": [
        float(ttc_df["p50_session_to_view_hrs"].iloc[0]),
        float(ttc_df["p50_view_to_cart_hrs"].iloc[0]),
        float(ttc_df["p50_cart_to_checkout_hrs"].iloc[0]),
        float(ttc_df["p50_checkout_to_purchase_hrs"].iloc[0]),
    ],
})

fig_ttc = px.bar(
    ttc_steps, x="transition", y="median_hours",
    title="Median Time Between Funnel Steps (Converting Sessions)",
    labels={"median_hours": "Median hours", "transition": ""},
    color="median_hours",
    color_continuous_scale="Blues",
    text="median_hours",
)
fig_ttc.update_traces(texttemplate="%{text:.2f}h", textposition="outside")
fig_ttc.update_layout(height=400, showlegend=False, coloraxis_showscale=False)
fig_ttc.show()


## 7. Shoppers funnel — alternative entry at `view_item`
Starting at `view_item` removes sessions that bounced before ever seeing a product.
This separates **shoppers** (people with product intent) from all traffic,
and gives the merchandising/PDP team a cleaner conversion rate to own.

**Hypothesis:** if this funnel shows a materially higher `add_to_cart` rate than the
full funnel, the main problem is getting people to products — not the product page itself.

In [14]:
# ── Section 7: Shoppers funnel — entry at view_item (BigQuery) ────────────
# Alternative funnel: only sessions where the user viewed a product.
# Removes "bounced" sessions that never reached a PDP — isolates shopping intent.
shoppers_sql = f"""
WITH base AS (
  SELECT
    user_pseudo_id,
    (SELECT ep.value.int_value FROM UNNEST(event_params) AS ep
     WHERE ep.key = 'ga_session_id' LIMIT 1) AS session_id,
    event_name,
    event_timestamp
  FROM `{DATASET}.events_*`
  WHERE
    _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name IN ('view_item','add_to_cart','begin_checkout','purchase')
),
session_events AS (
  SELECT user_pseudo_id, session_id, event_name, MIN(event_timestamp) AS first_ts
  FROM base
  WHERE session_id IS NOT NULL
  GROUP BY user_pseudo_id, session_id, event_name
),
pivoted AS (
  SELECT
    user_pseudo_id, session_id,
    MAX(CASE WHEN event_name = 'view_item'      THEN first_ts END) AS ts_view,
    MAX(CASE WHEN event_name = 'add_to_cart'    THEN first_ts END) AS ts_cart,
    MAX(CASE WHEN event_name = 'begin_checkout' THEN first_ts END) AS ts_checkout,
    MAX(CASE WHEN event_name = 'purchase'       THEN first_ts END) AS ts_purchase
  FROM session_events
  GROUP BY user_pseudo_id, session_id
)
SELECT
  COUNT(*)                                                                        AS step1_view_item,
  COUNTIF(ts_cart     IS NOT NULL AND ts_cart     >= ts_view)                    AS step2_add_to_cart,
  COUNTIF(ts_checkout IS NOT NULL AND ts_checkout >= ts_cart
                                 AND ts_cart      >= ts_view)                    AS step3_begin_checkout,
  COUNTIF(ts_purchase IS NOT NULL AND ts_purchase >= ts_checkout
                                 AND ts_checkout  >= ts_cart
                                 AND ts_cart      >= ts_view)                    AS step4_purchase
FROM pivoted
WHERE ts_view IS NOT NULL
"""

shoppers_raw = client.query(shoppers_sql).to_dataframe()
print("Shoppers funnel (entry at view_item):")
display(shoppers_raw)


Shoppers funnel (entry at view_item):


,step1_view_item,step2_add_to_cart,step3_begin_checkout,step4_purchase
0,77020,15146,5293,2751


In [15]:
# ── Section 7b: Shoppers funnel metrics + comparison ──────────────────────
shopper_labels = ["1. View Item", "2. Add to Cart", "3. Begin Checkout", "4. Purchase"]
shopper_cols   = ["step1_view_item","step2_add_to_cart","step3_begin_checkout","step4_purchase"]

shoppers_df = pd.DataFrame({
    "step":     shopper_labels,
    "sessions": [int(shoppers_raw[c].iloc[0]) for c in shopper_cols],
})
shoppers_df["step_cvr"]    = (shoppers_df["sessions"] / shoppers_df["sessions"].shift(1) * 100).round(1)
shoppers_df["overall_cvr"] = (shoppers_df["sessions"] / shoppers_df["sessions"].iloc[0] * 100).round(1)
shoppers_df["drop_off"]    = (100 - shoppers_df["step_cvr"]).round(1)

print("Shoppers funnel conversion rates:")
display(shoppers_df)

fig_shop = px.funnel(
    shoppers_df, x="sessions", y="step",
    title="Shoppers Funnel (Entry: View Item)",
    color_discrete_sequence=["#54A24B"],
)
for i, row in shoppers_df.iterrows():
    if i == 0:
        continue
    fig_shop.add_annotation(
        x=row["sessions"], y=row["step"],
        text=f"  ↓ {row['drop_off']}% drop-off  ({row['step_cvr']}% pass)",
        showarrow=False, xanchor="left",
        font=dict(size=11, color="#d62728"),
    )
fig_shop.update_layout(height=400, margin=dict(l=160, r=260, t=60, b=40))
fig_shop.show()

# Compare add-to-cart from session start vs from product viewers (diagnostic split).
full_cart_rate    = funnel_df.loc[funnel_df["step"]=="3. Add to Cart", "overall_cvr"].values[0]
shopper_cart_rate = shoppers_df.loc[shoppers_df["step"]=="2. Add to Cart", "step_cvr"].values[0]
print(f"
Add-to-cart rate from session start:  {full_cart_rate:.1f}%")
print(f"Add-to-cart rate from view_item:       {shopper_cart_rate:.1f}%")
print(f"
If these are similar → the PDP is the bottleneck.")
print(f"If the shoppers rate is much higher → the problem is getting users to products.")


Shoppers funnel conversion rates:


,step,sessions,step_cvr,overall_cvr,drop_off
0,1. View Item,77020,NaN,100.0,NaN
1,2. Add to Cart,15146,19.7,19.7,80.3
2,3. Begin Checkout,5293,34.9,6.9,65.1
3,4. Purchase,2751,52.0,3.6,48.0



Add-to-cart rate from session start:  4.2%
Add-to-cart rate from view_item:       19.7%

If these are similar → the PDP is the bottleneck.
If the shoppers rate is much higher → the problem is getting users to products.


## 8. Weekly cohort retention
Groups users by the week of their first `session_start`, then tracks what fraction
return in each subsequent week across the full Nov 2020–Jan 2021 window.
More cohorts = more reliable Week-1/2 retention estimates.

In [16]:
# ── Section 8: Weekly cohort retention (BigQuery) ───────────────────────────
# Cohort = calendar week (Monday) of each user's first session_start.
# Retention = distinct users with session_start in week N after cohort week.
cohort_sql = f"""
WITH first_sessions AS (
  SELECT
    user_pseudo_id,
    DATE_TRUNC(PARSE_DATE('%Y%m%d', MIN(event_date)), WEEK(MONDAY)) AS cohort_week
  FROM `{DATASET}.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name = 'session_start'
  GROUP BY user_pseudo_id
),
activity AS (
  SELECT DISTINCT
    user_pseudo_id,
    DATE_TRUNC(PARSE_DATE('%Y%m%d', event_date), WEEK(MONDAY)) AS activity_week
  FROM `{DATASET}.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '{DATE_START}' AND '{DATE_END}'
    AND event_name = 'session_start'
)
SELECT
  f.cohort_week,
  DATE_DIFF(a.activity_week, f.cohort_week, WEEK) AS weeks_since_cohort,
  COUNT(DISTINCT a.user_pseudo_id)                 AS active_users
FROM first_sessions f
JOIN activity a USING (user_pseudo_id)
WHERE a.activity_week >= f.cohort_week
GROUP BY f.cohort_week, weeks_since_cohort
ORDER BY f.cohort_week, weeks_since_cohort
"""

cohort_df = client.query(cohort_sql).to_dataframe()
print(f"Cohort data rows: {len(cohort_df)}")
display(cohort_df.head(15))


Cohort data rows: 105


,cohort_week,weeks_since_cohort,active_users
0,2020-10-26,0,2339
1,2020-10-26,1,251
2,2020-10-26,2,82
3,2020-10-26,3,56
4,2020-10-26,4,31
5,2020-10-26,5,24
6,2020-10-26,6,22
7,2020-10-26,7,5
8,2020-10-26,8,14
9,2020-10-26,9,5


In [17]:
# ── Section 8b: Retention heatmap ─────────────────────────────────────────
# Pivot: rows = acquisition week, columns = weeks since first session.
cohort_pivot = (
    cohort_df
    .pivot(index="cohort_week", columns="weeks_since_cohort", values="active_users")
    .fillna(0).astype(int)
)
# Normalize each cohort row by Week 0 (100% = everyone in acquisition week).
cohort_pct = cohort_pivot.div(cohort_pivot[0], axis=0).multiply(100).round(1)
cohort_pct.index   = cohort_pct.index.astype(str)
cohort_pct.columns = [f"Week {c}" for c in cohort_pct.columns]

print("Retention matrix (% of cohort returning):")
display(cohort_pct)

z    = cohort_pct.values.tolist()
txt  = [[f"{v:.0f}%" for v in row] for row in z]

fig_ret = go.Figure(go.Heatmap(
    z=z, x=cohort_pct.columns.tolist(), y=cohort_pct.index.tolist(),
    text=txt, texttemplate="%{text}",
    colorscale="Blues", reversescale=True, zmin=0, zmax=100,
))
fig_ret.update_layout(
    title="Weekly Cohort Retention (% returning)",
    xaxis_title="Weeks Since First Session",
    yaxis_title="Cohort Week",
    height=500,
    yaxis_autorange="reversed",
)
fig_ret.show()

# Headline retention KPIs averaged across all cohort weeks in the window.
if "Week 1" in cohort_pct.columns:
    print(f"
Avg Week-1 retention: {cohort_pct['Week 1'].mean():.1f}%")
if "Week 2" in cohort_pct.columns:
    print(f"Avg Week-2 retention: {cohort_pct['Week 2'].mean():.1f}%")


Retention matrix (% of cohort returning):


,Week 0,Week 1,Week 2,Week 3,Week 4,Week 5,Week 6,Week 7,Week 8,Week 9,Week 10,Week 11,Week 12,Week 13
cohort_week,,,,,,,,,,,,,,
2020-10-26,100.0,10.7,3.5,2.4,1.3,1.0,0.9,0.2,0.6,0.2,0.4,0.3,0.3,0.3
2020-11-02,100.0,6.2,3.4,2.3,2.3,1.9,1.3,0.6,0.4,0.7,0.5,0.4,0.4,0.0
2020-11-09,100.0,6.7,3.0,2.6,2.4,1.5,0.8,0.4,0.6,0.5,0.5,0.4,0.0,0.0
2020-11-16,100.0,5.7,3.3,2.4,1.8,0.7,0.4,0.5,0.5,0.4,0.3,0.0,0.0,0.0
2020-11-23,100.0,5.0,2.5,1.5,0.7,0.4,0.5,0.5,0.4,0.3,0.0,0.0,0.0,0.0
2020-11-30,100.0,5.3,2.2,0.9,0.6,0.7,0.5,0.4,0.4,0.0,0.0,0.0,0.0,0.0
2020-12-07,100.0,3.3,0.9,0.7,1.0,0.6,0.4,0.4,0.0,0.0,0.0,0.0,0.0,0.0
2020-12-14,100.0,2.9,0.9,1.2,0.9,0.6,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-12-21,100.0,2.5,1.1,1.0,0.6,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Avg Week-1 retention: 4.4%
Avg Week-2 retention: 1.7%


## 9. Summary & hypotheses

| Metric | What to look for | Hypothesis if worse than expected |
|---|---|---|
| **Session → View Item** | Should be the biggest absolute drop | Landing page / traffic quality; weak homepage routing to products |
| **View Item → Add to Cart** | Second-biggest lever | PDP UX, pricing, stock, missing trust signals |
| **Add to Cart → Purchase** | Usually high (>50%) for e-comm | Checkout friction, shipping cost surprise, payment issues |
| **Shoppers funnel ATC rate** | Compare to full-funnel ATC | If similar → PDP is bottleneck; if much higher → navigation/discovery is bottleneck |
| **AOV / median order value** | Baseline for revenue modelling | Segment by device or traffic source to find high-value cohorts |
| **Time-to-convert** | Checkout → Purchase should be <5 min | If long, investigate payment flow or shipping calculator UX |
| **Week-1 retention** | Benchmark: ~20%+ is healthy for e-comm | No email/push, weak remarketing, one-and-done campaigns |
| **Mobile vs desktop** | Expect mobile to lag at checkout | If similar, focus on merchandising & product discovery, not checkout |

---

### Next steps

**Point at real data:** swap `bigquery-public-data.ga4_obfuscated_sample_ecommerce` for your own
GA4 BigQuery export dataset. The SQL patterns are identical.

**Add segments:** extend any query with `traffic_source.medium`, `geo.country`,
or a `new_vs_returning` flag (`user_first_touch_timestamp` close to session start = new user).

**Statistical rigour:** for device or period comparisons, add a proportion Z-test
to confirm differences are real and not sampling noise — especially important
when tablet N is small.

## 10. AI-Powered Insight Generation

Pass all analysis results to an LLM to produce a **product-decision-ready brief** —
the kind of output a Product Analyst shares with a squad after completing an analysis.

This section demonstrates three things that map directly to modern product analytics roles:

- **Prompt engineering** — a structured system prompt with an explicit output schema
- **Contextualised LLM calls** — serialising DataFrame results as structured JSON context
- **AI-assisted insight workflows** — replacing manual write-up time with a reproducible, auditable step

> **API key required:** get one at [console.anthropic.com](https://console.anthropic.com).
> Set `ANTHROPIC_API_KEY` as an environment variable or paste it below.
> Never commit a key to git — add `.env` to `.gitignore`.

> **API key required.** This section calls an LLM to generate the product brief.
> - **Anthropic Claude** (paid): get a key at [console.anthropic.com](https://console.anthropic.com) → set `ANTHROPIC_API_KEY`
> - **Google Gemini** (free): get a key at [aistudio.google.com](https://aistudio.google.com) → see Section 10f below for a drop-in alternative
> - **Skip / pre-generated**: a sample brief (`product_brief_ga4_funnel.md`) is included in the repo so `generate_dashboard.py` works out of the box


In [18]:
# ── Section 10a: Install SDK(s) ─────────────────────────────────────────────
# Install the Anthropic SDK (used in 10d). If you prefer the free Gemini
# alternative, also install google-generativeai (see Section 10f).
%pip install -q anthropic
# %pip install -q google-generativeai  # ← uncomment for the free Gemini path


Note: you may need to restart the kernel to use updated packages.


In [19]:
# ── Section 10b: API key ───────────────────────────────────────────────────
import os

# Recommended: set in your shell before launching Jupyter
#   export ANTHROPIC_API_KEY="sk-ant-..."
# Fallback: paste key directly here for local use only (never commit).
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")

if ANTHROPIC_API_KEY == "YOUR_API_KEY_HERE":
    print("Set ANTHROPIC_API_KEY before running Section 10.")
else:
    print("API key loaded.")


Set ANTHROPIC_API_KEY before running Section 10.


In [20]:
# ── Section 10c: Compile analysis context for the LLM ─────────────────────
import json

def _f(val):
    """Safely convert a pandas/numpy scalar to a rounded Python float (NaN → None)."""
    try:
        v = float(val)
        return None if v != v else round(v, 2)  # NaN → None
    except (TypeError, ValueError):
        return None

# ctx: structured dict passed as JSON in the user prompt to Claude.
ctx = {}

# Session funnel (Section 3) — requires funnel_df from prior cells.
try:
    ctx["session_funnel"] = [
        {
            "step": r["step"],
            "sessions": int(r["sessions"]),
            "step_conversion_pct": _f(r["step_cvr"]),
            "overall_conversion_pct": _f(r["overall_cvr"]),
            "drop_off_pct": _f(r["drop_off"]),
        }
        for _, r in funnel_df.iterrows()
    ]
except NameError:
    print("funnel_df not found — run Section 3 first")

# Revenue summary (Section 4).
try:
    rv = revenue_df.iloc[0]
    ctx["revenue"] = {
        "purchasing_sessions": int(rv["purchasing_sessions"]),
        "total_revenue_usd": _f(rv["total_revenue"]),
        "avg_order_value_usd": _f(rv["avg_order_value"]),
        "median_order_value_usd": _f(rv["median_order_value"]),
    }
except NameError:
    print("revenue_df not found — run Section 4 first")

# Device breakdown at purchase step (Section 5).
try:
    purchase_rows = device_long[device_long["step_col"] == "step5_purchase"]
    ctx["purchase_rate_by_device"] = [
        {
            "device": r["device_category"],
            "sessions": int(r["sessions"]),
            "pct_of_device_sessions": _f(r["pct"]),
        }
        for _, r in purchase_rows.iterrows()
    ]
except NameError:
    print("device_long not found — run Section 5 first")

# Median hours between funnel steps (Section 6).
try:
    ctx["median_time_between_steps_hours"] = {
        "session_to_view_item": _f(ttc_df["p50_session_to_view_hrs"].iloc[0]),
        "view_item_to_add_to_cart": _f(ttc_df["p50_view_to_cart_hrs"].iloc[0]),
        "add_to_cart_to_checkout": _f(ttc_df["p50_cart_to_checkout_hrs"].iloc[0]),
        "checkout_to_purchase": _f(ttc_df["p50_checkout_to_purchase_hrs"].iloc[0]),
        "total_session_to_purchase": _f(ttc_df["p50_total_hrs"].iloc[0]),
    }
except NameError:
    print("ttc_df not found — run Section 6 first")

# Shoppers vs full funnel add-to-cart comparison (Section 7).
try:
    full_cart = funnel_df.loc[funnel_df["step"] == "3. Add to Cart", "overall_cvr"].values
    shop_cart = shoppers_df.loc[shoppers_df["step"] == "2. Add to Cart", "step_cvr"].values
    ctx["shoppers_funnel"] = {
        "note": "Funnel starting at view_item — removes sessions that never saw a product",
        "add_to_cart_rate_from_all_sessions_pct": _f(full_cart[0]) if len(full_cart) else None,
        "add_to_cart_rate_from_product_viewers_pct": _f(shop_cart[0]) if len(shop_cart) else None,
    }
except NameError:
    print("shoppers_df not found — run Section 7 first")

# Cohort retention averages (Section 8).
try:
    wk1 = cohort_pct["Week 1"].mean() if "Week 1" in cohort_pct.columns else None
    wk2 = cohort_pct["Week 2"].mean() if "Week 2" in cohort_pct.columns else None
    ctx["cohort_retention"] = {
        "avg_week1_return_pct": _f(wk1),
        "avg_week2_return_pct": _f(wk2),
        "note": "% of users who had any session in week N after their acquisition week",
    }
except NameError:
    print("cohort_pct not found — run Section 8 first")

print("Context compiled — passing to LLM:")
# default=str avoids crashes if any value is still a numpy/pandas scalar
print(json.dumps(ctx, indent=2, default=str))


Context compiled — passing to LLM:
{
  "session_funnel": [
    {
      "step": "1. Session Start",
      "sessions": 354857,
      "step_conversion_pct": null,
      "overall_conversion_pct": 100.0,
      "drop_off_pct": null
    },
    {
      "step": "2. View Item",
      "sessions": 75261,
      "step_conversion_pct": 21.2,
      "overall_conversion_pct": 21.2,
      "drop_off_pct": 78.8
    },
    {
      "step": "3. Add to Cart",
      "sessions": 14870,
      "step_conversion_pct": 19.8,
      "overall_conversion_pct": 4.2,
      "drop_off_pct": 80.2
    },
    {
      "step": "4. Begin Checkout",
      "sessions": 5205,
      "step_conversion_pct": 35.0,
      "overall_conversion_pct": 1.5,
      "drop_off_pct": 65.0
    },
    {
      "step": "5. Purchase",
      "sessions": 2721,
      "step_conversion_pct": 52.3,
      "overall_conversion_pct": 0.8,
      "drop_off_pct": 47.7
    }
  ],
  "revenue": {
    "purchasing_sessions": 2461,
    "total_revenue_usd": 193678.0,
    "av

In [28]:
# ── Section 10d: Generate product brief via Claude ─────────────────────────
import anthropic

# System prompt: fixed output schema + product-analyst tone (not data-science jargon).
SYSTEM_PROMPT = """You are a senior Product Analyst embedded in a cross-functional product squad.
Your job is to translate funnel analysis results into clear, decision-ready product briefs.

Always produce exactly these five sections:

## Executive Summary
2-3 sentences. State the single most important finding and its business implication.

## Key Findings
3-5 numbered findings. Each must include a specific number from the data and a
"so what" that a product manager can act on.

## Hypotheses
3 prioritised hypotheses that could explain the patterns.
Label each: [HIGH / MEDIUM / LOW] confidence based on how strongly the data supports it.

## Recommended Actions
3 specific, testable actions. For each state: what to do, why, and what metric to watch.

## What to Measure Next
2-3 analyses that would either confirm these hypotheses or reveal what this analysis misses.

Rules:
- Be opinionated. Choose a priority — do not hedge everything equally.
- Write for a product manager, not a data scientist. No statistical jargon.
- Flag explicitly where findings come from a public sample and need validation on real data.
- Where relevant, note how e-commerce metrics translate to B2B SaaS equivalents
  (e.g. session_start → login, view_item → feature discovery, purchase → activation)."""

# User prompt embeds compiled ctx JSON from the previous cell.
USER_PROMPT = f"""Produce a product brief from these funnel analysis results.

Dataset: GA4 public e-commerce sample (Nov 2020 – Jan 2021, session-scoped funnel).
The methodology is directly transferable to B2B SaaS — where "purchase" maps to
a key activation event such as completing onboarding or adopting a core feature.

{json.dumps(ctx, indent=2)}"""


def generate_brief(api_key: str) -> str:
    """Call Claude with system + user prompts; return markdown brief text."""
    client = anthropic.Anthropic(api_key=api_key)
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1800,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": USER_PROMPT}],
    )
    return response.content[0].text


if ANTHROPIC_API_KEY != "YOUR_API_KEY_HERE":
    print("Calling Claude... (~10-20 seconds)\n")
    brief = generate_brief(ANTHROPIC_API_KEY)
    print(brief)
else:
    brief = None
    print("Skipped — set ANTHROPIC_API_KEY first.")


Skipped — set ANTHROPIC_API_KEY first.


In [25]:
# ── Section 10e: Display and export product brief ─────────────────────────
from IPython.display import Markdown, display
from pathlib import Path
import pandas as pd

if brief:
    # Render Claude output as formatted markdown inside the notebook.
    display(Markdown(brief))

    # Persist to repo root for README, portfolio, or sharing outside Jupyter.
    header = (
        "# Product Brief — GA4 Funnel Analysis

"
        f"*Generated {pd.Timestamp.now().strftime('%Y-%m-%d')} · "
        "Session-scoped funnel · Nov 2020 – Jan 2021 · "
        "[GA4 public e-commerce sample](https://console.cloud.google.com/marketplace/"
        "product/google-analytics-public-data/ga4_obfuscated_sample_ecommerce)*

"
        "---

"
    )
    out = Path("product_brief_ga4_funnel.md")
    out.write_text(header + brief, encoding="utf-8")
    print(f"Brief saved to {out}")


In [ ]:
# ── Section 10f: FREE ALTERNATIVE — Google Gemini ───────────────────────────
# Gemini 1.5 Flash is free for personal use via Google AI Studio (no credit card).
#
# Setup (one-time):
#   1. Go to https://aistudio.google.com → "Get API key" (free, just needs a Google account)
#   2. pip install google-generativeai
#   3. Uncomment the lines below and paste your key (or use env var GEMINI_API_KEY)
#
# This cell is a drop-in replacement for Section 10d.
# Run EITHER 10d (Anthropic) OR 10f (Gemini) — not both.

# import os
# import google.generativeai as genai
#
# GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_KEY_HERE")
# genai.configure(api_key=GEMINI_API_KEY)
#
# model = genai.GenerativeModel(
#     model_name="gemini-1.5-flash",
#     system_instruction=SYSTEM_PROMPT,   # defined in Section 10d
# )
#
# response = model.generate_content(USER_PROMPT)  # USER_PROMPT from Section 10d
# brief = response.text
# print(brief)
